In [10]:
import pandas as pd
from chromadb.utils.batch_utils import create_batches
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [3]:
df = pd.read_csv("/Users/kishohars/Projects/rag_qa_football/data/ea_fc26_players.csv")
df.head()

,id,rank,overallRating,firstName,lastName,commonName,birthdate,height,weight,skillMoves,...,headingAccuracy,aggression,jumping,stamina,strength,gkDiving,gkHandling,gkKicking,gkPositioning,gkReflexes
0,209331,1,91,Mohamed,Salah,NaN,6/15/1992 12:00:00 AM,175,72,4,...,59,63,79,88,75,14,14,9,11,14
1,231747,3,91,Kylian,Mbappé,NaN,12/20/1998 12:00:00 AM,182,75,5,...,78,61,90,83,77,13,5,7,11,6
2,231443,5,90,Ousmane,Dembélé,NaN,5/15/1997 12:00:00 AM,178,67,5,...,74,58,84,76,69,6,6,14,10,13
3,231866,6,90,Rodrigo,Hernández Cascante,Rodri,6/22/1996 12:00:00 AM,190,82,3,...,81,85,83,91,83,10,10,7,14,8
4,203376,8,90,Virgil,van Dijk,NaN,7/8/1991 12:00:00 AM,193,92,2,...,88,85,89,75,93,13,10,13,11,11


In [5]:
df_columns = df.columns

In [26]:
# 1. Load the CSV file
loader = CSVLoader(file_path="/Users/kishohars/Projects/rag_qa_football/data/ea_fc26_players.csv")
data = loader.load()

# 2. Initialize the splitter
# Adjust chunk_size and chunk_overlap based on your requirements
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

# 3. Split the loaded documents
docs = splitter.split_documents(data)

print(f"Number of chunks: {len(docs)}")

Number of chunks: 16254


In [9]:
# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Extract text from Document objects
sentences = [doc.page_content for doc in docs]

# 3. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

# 4. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10607.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(16254, 384)
tensor([[1.0000, 0.8591, 0.8263,  ..., 0.7928, 0.8022, 0.8117],
        [0.8591, 1.0000, 0.9164,  ..., 0.8488, 0.8302, 0.8635],
        [0.8263, 0.9164, 1.0000,  ..., 0.7891, 0.7844, 0.8012],
        ...,
        [0.7928, 0.8488, 0.7891,  ..., 1.0000, 0.9480, 0.9490],
        [0.8022, 0.8302, 0.7844,  ..., 0.9480, 1.0000, 0.9543],
        [0.8117, 0.8635, 0.8012,  ..., 0.9490, 0.9543, 1.0000]])


In [11]:
chroma_client = chromadb.Client()

In [12]:
collection = chroma_client.create_collection(name="football")

In [30]:
batches = create_batches(api=chroma_client, 
                        embeddings=embeddings, 
                        ids=[str(i) for i in range(len(embeddings))]
                        )

for batch in batches:
    ids_batch, embeddings_batch, _, _ = batch
    collection.add(
        ids=ids_batch,
        embeddings=embeddings_batch
    )

print(f"Added {len(embeddings)} documents in {len(batches)} batches")

Added 16254 documents in 3 batches
